# Milestone 5 lab, notebook 1 of 7: A clock is not enough

## Where we are

| Notebook | What you do |
|---|---|
| **1. A clock is not enough** (you are here) | Start two steps by the clock, and get yesterday's report |
| 2. Tasks and arrows | Read the DAG, and run it once |
| 3. Which day is this run for? | The logical date, the data interval, and a run you trigger by hand |
| 4. When a task fails | Break the API, then break the data, and watch what Airflow does |
| 5. The days you missed | Run a week with one command. Run one day again |
| 6. Every five minutes | A lab: the same pipeline as a micro-batch |
| 7. The bad morning | The drill: the API is down, then a page comes twice |

**Before you start,** open [`milestone_5_flow.html`](../milestone_5_flow.html) in a browser.

This milestone needs three things running in Docker: the warehouse, the order system, and Airflow. One command starts them all, from the project folder:

```
docker compose --profile schedule up -d --build
```

The first time, it downloads about 3 GB and takes several minutes. If you still have `uv run python -m order_system` running in a terminal from earlier milestones, stop it first: the order system now runs in Docker, on the same port.

In [1]:
from lab_helpers import check_api, dbt, lab_warehouse, set_morning, show_doc, table

from daily_orders import pipeline
from daily_orders.report import format_answer

check_api()

The order system is running. Its clock says 2026-06-10 07:00:00 in Arden.


## Step 1: The pipeline is two steps now

Milestone 4 ended with one command that did everything. To put a pipeline on a schedule, it helps to cut it into steps that can be started, watched and repeated one by one. The program has three:

| Step | What it does |
|---|---|
| `extract-load` | Reads what changed in the order system, and lands it raw |
| `transform-test` | Runs dbt: builds the models and tests the tables |
| `report` | Prints the manager's report |

Run Monday, step by step.

In [2]:
con = lab_warehouse(fresh=True)

set_morning("2026-06-08")
loaded = pipeline.extract_and_load(pipeline.httpx.Client(base_url="http://127.0.0.1:8010"), con)
print(f"extract-load: run {loaded['run_id']} landed {loaded['rows_landed']} rows.")

done = pipeline.transform_and_test(con)
print(f"transform-test: dbt built {done.models_built} models and passed {done.tests_passed} tests.\n")
print(pipeline.report_text(con))

It is now Monday 2026-06-08 07:00:00 in Arden.
extract-load: run 2026-06-08_1 landed 226 rows.


transform-test: dbt built 7 models and passed 31 tests.

Arden daily orders. Report for the morning of Mon 2026-06-08

  Yesterday   Sun 2026-06-07    30 orders      $807.11
  Day before  Sat 2026-06-06    33 orders      $786.71
  Change                         -3 orders      +$20.40

  Earlier days: no change in this run.


## Step 2: Start them by the clock

The simplest way to run something every morning is **cron**: a tool on every Linux and macOS computer that starts a command at a set time. A cron **schedule** is five fields:

```
┌ minute   ┌ hour   ┌ day of month   ┌ month   ┌ day of week
0          7        *                *         *            = every day at 07:00
```

So a first plan is two lines. The load at 07:00. The transform five minutes later, to be safe.

```
0 7 * * *    python -m daily_orders extract-load
5 7 * * *    python -m daily_orders transform-test
```

On most mornings the load takes a few seconds, and this works. Now play a morning when the order system is slow, and the load is still running at 07:05.

It is Tuesday. The clock reaches 07:05. Cron starts the transform. **The load has not finished.**

In [3]:
set_morning("2026-06-09")

# 07:05. cron starts the transform. The load of this morning has not landed anything yet.
done = pipeline.transform_and_test(con)
print(f"transform-test: dbt built {done.models_built} models and passed {done.tests_passed} tests.\n")
print(pipeline.report_text(con))

It is now Tuesday 2026-06-09 07:00:00 in Arden.


transform-test: dbt built 7 models and passed 31 tests.

Arden daily orders. Report for the morning of Mon 2026-06-08

  Yesterday   Sun 2026-06-07    30 orders      $807.11
  Day before  Sat 2026-06-06    33 orders      $786.71
  Change                         -3 orders      +$20.40

  Earlier days: no change in this run.


Read that carefully.

- dbt ran, and **every test passed**.
- The report says "morning of Mon 2026-06-08". It is Tuesday.

The transform found Monday's rows, because Tuesday's were not there yet. They are perfectly valid rows. Nothing failed. The manager opens the report at 08:00 and reads yesterday's numbers.

A few minutes later the load finishes.

In [4]:
loaded = pipeline.extract_and_load(pipeline.httpx.Client(base_url="http://127.0.0.1:8010"), con)
print(f"extract-load: run {loaded['run_id']} landed {loaded['rows_landed']} rows. Nobody transforms them until tomorrow.")

table(con, "SELECT max(order_date) AS newest_day_in_the_report FROM marts.daily_orders")

extract-load: run 2026-06-09_1 landed 102 rows. Nobody transforms them until tomorrow.


,newest_day_in_the_report
0,2026-06-07


Tuesday's rows are in the raw table now. The report still ends on Sunday, and it will until cron starts the transform again, tomorrow at 07:05.

## Step 3: What was missing

Cron knew the **time**. It did not know that the transform **needs** the load to have finished. The real rule was never "start at 07:05". It was "start when the load is done".

| Morning | With two cron lines |
|---|---|
| The load takes 2 minutes | Fine |
| The load takes 8 minutes | The transform publishes yesterday's report, and every test passes |
| The load fails | The transform still starts. Nobody is told |

In [5]:
show_doc("HLD.md", "3. Why a clock is not enough")

**From `docs/HLD.md`**

> ## 3. Why a clock is not enough
> 
> The simplest schedule is **cron**: a tool that starts a command at a set time.
> 
> ```
> 0 7 * * *    python -m daily_orders extract-load
> 5 7 * * *    python -m daily_orders transform-test
> ```
> 
> | Morning | What happens |
> |---|---|
> | The load takes 2 minutes | Fine. The transform starts at 07:05 and finds today's rows |
> | The load takes 8 minutes | The transform starts at 07:05, finds **yesterday's** rows, passes every test, and publishes yesterday's report |
> | The load fails | The transform still starts. Nobody is told |
> 
> Cron knows the time. It does not know that one step **needs** another. An orchestrator does.
> 
> | Question | Cron | An orchestrator |
> |---|---|---|
> | Does a step wait for the step before it? | No | Yes |
> | Does it try again after a failure? | No | Yes, as often as you say |
> | Can you see what ran, and when? | Only if you wrote a log | Yes |
> | Can it run last Tuesday again? | By hand | One command |

In [6]:
con.close()

A tool that runs steps in the right order, at the right time, and watches them, is called an **orchestrator**. This project uses Apache Airflow. The rest of this milestone is about it.

## New words

| Word | Meaning | Here |
|---|---|---|
| **Schedule** | "A schedule starts a run at a time." | Every day at 07:00 |
| **Cron** | "Cron is a Linux tool that starts a command at a set time." Its five fields are also the usual way to write a schedule | `0 7 * * *` |
| **Orchestration** | Running the steps of a pipeline in the right order, at the right time, and watching them. "An orchestrator is a tool that runs tasks in the right order and watches them." | Airflow, from the next notebook on |

The definitions are from the D8Loop chapter [Orchestration & Scheduling](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/orchestration-and-scheduling).

## What you now know

- A schedule starts a run at a time. `0 7 * * *` means every day at 07:00.
- Two commands started by the clock do not wait for each other.
- When the load was slow, the transform published yesterday's report and passed every test.
- The real rule is a dependency: this step needs that step to finish first.

**Say it in an interview:** "Cron can start a job on time, but it cannot express that one step needs another. With two cron lines five minutes apart, a slow load made my transform publish yesterday's numbers, with every test green. That is the reason for an orchestrator: dependencies, retries, and a history of runs."

Next: [notebook 2](02_tasks_and_arrows.ipynb) writes that dependency down.